# Nanonets-OCR2-3B

In [1]:
!pip install -U torch torchvision transformers pdf2image pillow
!apt-get update
!apt-get install -y poppler-utils


Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,867 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,600 kB]
Get:14 http://ar

In [2]:
import torch
from transformers import AutoProcessor, AutoModelForVision2Seq
from pdf2image import convert_from_path
from PIL import Image


In [3]:
import torch
from transformers import AutoProcessor, AutoModelForVision2Seq

MODEL_ID = "nanonets/Nanonets-OCR2-3B"

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Qwen2_5_VLForConditionalGeneration(
  (model): Qwen2_5_VLModel(
    (visual): Qwen2_5_VisionTransformerPretrainedModel(
      (patch_embed): Qwen2_5_VisionPatchEmbed(
        (proj): Conv3d(3, 1280, kernel_size=(2, 14, 14), stride=(2, 14, 14), bias=False)
      )
      (rotary_pos_emb): Qwen2_5_VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-31): 32 x Qwen2_5_VLVisionBlock(
          (norm1): Qwen2RMSNorm((1280,), eps=1e-06)
          (norm2): Qwen2RMSNorm((1280,), eps=1e-06)
          (attn): Qwen2_5_VLVisionAttention(
            (qkv): Linear(in_features=1280, out_features=3840, bias=True)
            (proj): Linear(in_features=1280, out_features=1280, bias=True)
          )
          (mlp): Qwen2_5_VLMLP(
            (gate_proj): Linear(in_features=1280, out_features=3420, bias=True)
            (up_proj): Linear(in_features=1280, out_features=3420, bias=True)
            (down_proj): Linear(in_features=3420, out_features=1280, bias=True)
            (act_fn): SiLUAc

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload() 
    pdf_path = list(uploaded.keys())[0]
    print("Using PDF:", pdf_path)

except ImportError:
    pdf_path = "data/benchmark/flexqube_arsredovisning_2022.pdf" 
    
    print(f"Running locally, using: {pdf_path}")

Saving FlexQube.pdf to FlexQube.pdf
PDF loaded: FlexQube.pdf


In [5]:
pages = convert_from_path(pdf_path, dpi=100)

print("Antal sidor:", len(pages))
print("Typ:", type(pages))


Antal sidor: 68
Typ: <class 'list'>


### Skala ner bilder

In [6]:
def downscale_image(img, max_side=1200):
    w, h = img.size
    scale = min(max_side / w, max_side / h, 1.0)
    if scale < 1.0:
        img = img.resize(
            (int(w * scale), int(h * scale)),
            Image.BICUBIC
        )
    return img


### OCR-prompt

In [7]:
PROMPT = (
    "Extract ALL text from this image exactly as it appears. "
    "Do NOT paraphrase, summarize, or reorder. "
    "Preserve tables, numbers, and headings."
)


### OCR-funktion

In [8]:
def run_ocr(image: Image.Image, prompt: str):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": prompt}
            ]
        }
    ]

    # Bygg korrekt chat-input med <image>-token
    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=text,
        images=image,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            num_beams=1,
            use_cache=False
        )

    decoded = processor.decode(output_ids[0], skip_special_tokens=True)

    # ta bort chat-metada
    if "assistant" in decoded:
        decoded = decoded.split("assistant", 1)[-1].strip()

    del inputs, output_ids
    torch.cuda.empty_cache()

    return decoded



In [9]:
results = []

for i, page in enumerate(pages):
    page = downscale_image(page)

    print(
        f"Processing page {i+1}/{len(pages)} | "
        f"size={page.size}"
    )

    text = run_ocr(page, PROMPT)
    results.append(text)

full_text = "\n\n".join(results)



The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Processing page 1/68 | size=(827, 1170)
Processing page 2/68 | size=(1200, 848)
Processing page 3/68 | size=(1200, 848)
Processing page 4/68 | size=(1200, 848)
Processing page 5/68 | size=(1200, 848)
Processing page 6/68 | size=(1200, 848)
Processing page 7/68 | size=(1200, 848)
Processing page 8/68 | size=(1200, 848)
Processing page 9/68 | size=(1200, 848)
Processing page 10/68 | size=(1200, 848)
Processing page 11/68 | size=(1200, 848)
Processing page 12/68 | size=(1200, 848)
Processing page 13/68 | size=(1200, 848)
Processing page 14/68 | size=(1200, 848)
Processing page 15/68 | size=(1200, 848)
Processing page 16/68 | size=(1200, 848)
Processing page 17/68 | size=(1200, 848)
Processing page 18/68 | size=(1200, 848)
Processing page 19/68 | size=(1200, 848)
Processing page 20/68 | size=(1200, 848)
Processing page 21/68 | size=(1200, 848)
Processing page 22/68 | size=(1200, 848)
Processing page 23/68 | size=(1200, 848)
Processing page 24/68 | size=(1200, 848)
Processing page 25/68 | s

In [14]:
print(full_text[:1500])


.
user
Extract ALL text from this image exactly as it appears. Do NOT paraphrase, summarize, or reorder. Preserve tables, numbers, and headings.
assistant
Årsredovisning
2022

FLEXQUBE

Cockpit
Mission Planner
Wait 8 Seconds
Go to Station Process 1

7

READY TO FOLLOW

FLEXQUBE®

.
user
Extract ALL text from this image exactly as it appears. Do NOT paraphrase, summarize, or reorder. Preserve tables, numbers, and headings.
assistant
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

.
user
Extract ALL text from this image exactly as